# Specialization cannot form a cycle.

Run the setup first, then each step in order. This notebook executes the same declarations, calls and assertions as the [script](37_error_cycle.py). Outputs were captured by direct Python execution in a fresh process. No Jupyter kernel was used.


In [ ]:
!pip install -q aoa-action-machine

In [2]:
from __future__ import annotations

from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import (
    BaseEnumEntity,
    BaseSpecEntity,
    Classifier,
    Generalization,
    Inverse,
    Specialization,
    value,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage


class MusicDomain(BaseDomain):
    """Group the cycle experiment."""

    name = "music"
    description = "Cycle experiment"


@entity(description="Kind of ASpecEntity", domain=MusicDomain)
class ASpecEntityKindEnumEntity(BaseEnumEntity):
    """Declare the codes the axis borrows."""

    B = value(code="b", display_name="B")


@entity(description="First level", domain=MusicDomain)
class ASpecEntity(BaseSpecEntity):
    """Declare the first level of detail."""

    id: str = Field(description="Identifier")
    detail: Annotated[Specialization[BSpecEntity, ASpecEntityKindEnumEntity], Inverse(field_name="back")] = Field(
        description="Next level"
    )
    back: Annotated[
        Generalization[BSpecEntity],
        Classifier(variant=BSpecEntityKindEnumEntity.A),
        Inverse(field_name="detail"),
    ] = Field(description="Return to B")


@entity(description="Kind of BSpecEntity", domain=MusicDomain)
class BSpecEntityKindEnumEntity(BaseEnumEntity):
    """Declare the codes the axis borrows."""

    A = value(code="a", display_name="A")


@entity(description="Second level", domain=MusicDomain)
class BSpecEntity(BaseSpecEntity):
    """Declare the second level of detail."""

    id: str = Field(description="Identifier")
    back: Annotated[
        Generalization[ASpecEntity],
        Classifier(variant=ASpecEntityKindEnumEntity.B),
        Inverse(ASpecEntity, "detail"),
    ] = Field(description="Return to A")
    detail: Annotated[Specialization[ASpecEntity, BSpecEntityKindEnumEntity], Inverse(field_name="back")] = Field(
        description="Next level"
    )


@meta(description="Stores the music entities")
@entities(
    ASpecEntityKindEnumEntity,
    ASpecEntity,
    BSpecEntityKindEnumEntity,
    BSpecEntity,
)
class MusicStorage(BaseStorage):
    """Hold the music entities."""


ASpecEntity.model_rebuild()
BSpecEntity.model_rebuild()

## Load the integration API


In [3]:
from aoa.action_machine.domain.exceptions import SpecializationDeclarationError

## Load the integration API


In [4]:
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations

## Check the expected refusal and assert its diagnostic


In [5]:
try:
    validate_entity_specializations()
except SpecializationDeclarationError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The broken declaration was accepted")

SpecializationDeclarationError: Specialization 'detail' on entity 'ASpecEntity' is invalid: the head and its alternatives form a cycle: ASpecEntity -> BSpecEntity -> ASpecEntity
